# VAR estructurales (2): restricciones de signo

El notebook A identificó con **ceros**: el choque de política no mueve al PBI ni a la inflación
dentro del trimestre. Es una afirmación fuerte, y produjo un *price puzzle*, es decir una inflación
que sube después de una contracción monetaria.

Las restricciones de signo imponen algo más débil y a menudo más creíble: la **dirección** del
efecto. Un choque contractivo sube la tasa y baja la inflación. Sobre el PBI no se impone nada,
porque es justamente lo que se quiere averiguar. Es la estrategia agnóstica de Uhlig (2005).

El precio de ese cambio es que la identificación deja de ser puntual: en vez de **una** matriz de
impacto queda un **conjunto** de matrices compatibles con los signos, y hay que reportar el
conjunto entero.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import BayesianVAR, set_style
import sys; sys.path.append("..")          # course_data.py vive en la raíz del repositorio
import course_data as cd

set_style()

levels = cd.load()                          # copia congelada de la base del curso
Y = cd.small_open_economy(levels)
n, p, H = Y.shape[1], 2, 20
labels = [cd.LABELS[c] for c in Y.columns]

model = BayesianVAR(Y, lags=p, prior_type=1, post_draws=4000, burnin=0.5,
                    hor=H + 1, seed=1, progress=False)
model.sample_posterior()
model.names = labels
irf_chol = model.compute_irfs()            # referencia recursiva del notebook A
print("base del curso, descargada el", cd.vintage())
print(f"{len(model.beta_draws)} extracciones | IRF recursivas {irf_chol.shape}")

## 1. Recorrer el conjunto identificado

Todo descansa en el conteo del notebook A. Si $L$ es el factor de Cholesky de $\Sigma$ y $Q$ es
ortonormal, entonces

$$\tilde{S} = LQ \quad\Longrightarrow\quad \tilde{S}\tilde{S}' = LQQ'L' = LL' = \Sigma,$$

de modo que **toda** $\tilde S$ así construida es un candidato válido: ajusta los datos exactamente
igual que la de Cholesky. El conjunto de candidatos se recorre sorteando $Q$ y quedándose con los
que cumplen los signos.

Con un solo choque de interés basta una columna. Si $q$ es un vector unitario, $\tilde s = Lq$ es
un vector de impacto admisible, y el conjunto de todos ellos es un elipsoide.

In [ ]:
def wold_weights(model, B, h):
    '''Psi_j = J F^j J' para j = 0, ..., h.'''
    F = model.build_companion_matrix(B, n, p)
    J = np.zeros((n, n * p)); J[:, :n] = np.eye(n)
    out = np.zeros((h + 1, n, n)); power = np.eye(n * p)
    for j in range(h + 1):
        out[j] = J @ power @ J.T
        power = power @ F
    return out

# Una extraccion fija, para ver la geometria sin ruido de estimacion.
d = 0
B_d = model.reshape_beta(model.beta_draws[d], model.ncoeff_eq, n)
L_d = np.linalg.cholesky(model.Sigma_draws[d])

rng = np.random.default_rng(3)
q = rng.standard_normal((4000, n))
q /= np.linalg.norm(q, axis=1, keepdims=True)
impactos = q @ L_d.T                       # cada fila es un vector de impacto admisible
ok = (impactos[:, 3] > 0) & (impactos[:, 2] < 0)   # tasa sube, inflacion baja

fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.scatter(impactos[~ok, 2], impactos[~ok, 3], s=4, color="#C9C0B6", label="descartados")
ax.scatter(impactos[ok, 2], impactos[ok, 3], s=4, color="#B0472A", label="admisibles")
ax.scatter(L_d[2, 3], L_d[3, 3], s=70, marker="*", color="#3D647F",
           zorder=5, label="Cholesky")
ax.axhline(0, color="#6B5F52", lw=0.7); ax.axvline(0, color="#6B5F52", lw=0.7)
ax.set_xlabel("impacto sobre la inflación"); ax.set_ylabel("impacto sobre la tasa")
ax.set_title("El conjunto de impactos compatibles con $\\Sigma$", fontsize=11)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout(); plt.show()

print(f"fracción del elipsoide que cumple los signos: {100 * ok.mean():.0f}%")

Cada punto gris o rojo es una economía distinta que ajusta los datos **exactamente igual**. La
estrella es la que eligió Cholesky. Los signos no seleccionan un punto: seleccionan un arco.

## 2. Un choque de política, a lo Uhlig

El algoritmo es el de Rubio-Ramírez, Waggoner y Zha (2010), en versión bayesiana: se repite para
cada extracción de la posterior, de modo que el resultado mezcla la incertidumbre de estimación
con la de identificación.

1. Tomar una extracción $(B^{(m)}, \Sigma^{(m)})$ de la posterior y calcular $L^{(m)}$.
2. Sortear $q$ uniforme en la esfera unitaria y formar $\tilde s = L^{(m)} q$.
3. Verificar los signos, permitiendo cambiar $q$ por $-q$, porque $\varepsilon$ y $-\varepsilon$
   son el mismo choque con el nombre cambiado.
4. Si se cumplen, guardar $\Theta_h = \Psi_h \tilde s$; si no, volver a sortear.

In [ ]:
def conjunto_politica(model, restricciones, max_intentos=100, seed=7):
    '''Un choque por extraccion: tasa sube, inflacion baja, PBI libre.'''
    rng = np.random.default_rng(seed)
    aceptados, intentos = [], 0
    for m in range(len(model.beta_draws)):
        B = model.reshape_beta(model.beta_draws[m], model.ncoeff_eq, n)
        L = np.linalg.cholesky(model.Sigma_draws[m])
        W = wold_weights(model, B, H)
        for _ in range(max_intentos):
            intentos += 1
            q = rng.standard_normal(n); q /= np.linalg.norm(q)
            s = L @ q
            if s[3] < 0:
                s = -s                       # normalizacion: la tasa sube
            if all(np.sign(s[fila]) == signo for fila, signo in restricciones):
                aceptados.append(W @ s)
                break
    return np.array(aceptados), intentos

# fila 3 = tasa, fila 2 = inflacion
conjunto, intentos = conjunto_politica(model, [(3, +1), (2, -1)])
print(f"aceptados {len(conjunto)} de {len(model.beta_draws)} extracciones "
      f"({100 * len(conjunto) / intentos:.0f}% de los sorteos)")

In [ ]:
def figura_conjunto(conjunto, referencia, titulo):
    med = np.median(conjunto, axis=0)
    q05, q95 = np.quantile(conjunto, 0.05, axis=0), np.quantile(conjunto, 0.95, axis=0)
    pasos = np.arange(conjunto.shape[1])
    fig, axes = plt.subplots(1, 4, figsize=(12, 3.1))
    for i, (ax, name) in enumerate(zip(axes, labels)):
        ax.fill_between(pasos, q05[..., i], q95[..., i], color="#B0472A", alpha=0.18,
                        lw=0, label="conjunto (5 a 95 %)")
        ax.plot(pasos, med[..., i], color="#B0472A", lw=1.6, label="mediana del conjunto")
        ax.plot(pasos, referencia[:, i], color="#3D647F", lw=1.2, ls=(0, (3, 2)),
                label="recursiva (Cholesky)")
        ax.axhline(0, color="#6B5F52", lw=0.7)
        ax.set_title(name, fontsize=10); ax.set_xlabel("trimestres")
    axes[0].legend(fontsize=8)
    fig.suptitle(titulo, fontsize=11)
    fig.tight_layout(); plt.show()

# La escala no es comparable entre esquemas, asi que ambos van en desviaciones estandar.
ref = np.median(irf_chol, axis=0)[:, :, 3]
figura_conjunto(conjunto, ref, "Choque contractivo de política monetaria")

In [ ]:
med = np.median(conjunto, axis=0)
print("Respuesta del PBI al choque contractivo")
print(f"{'h':>3} {'mediana':>9} {'5%':>8} {'95%':>8} {'% modelos con PBI<0':>22}")
for h in (0, 2, 4, 8, 12):
    lo, hi = np.quantile(conjunto[:, h, 1], [0.05, 0.95])
    share = 100 * (conjunto[:, h, 1] < 0).mean()
    print(f"{h:>3} {med[h,1]:>9.3f} {lo:>8.3f} {hi:>8.3f} {share:>21.0f}%")
print(f"\nreferencia recursiva en h=4: {ref[4,1]:.3f}")

Este es el resultado central, y conviene leerlo despacio.

Con la identificación recursiva el PBI caía de manera nítida. Con restricciones de signo, una vez
que se deja **libre** la respuesta del PBI, el conjunto de modelos admisibles no se pone de acuerdo
sobre su signo: alrededor de la mitad de los modelos aceptados implica una caída y la otra mitad,
un aumento.

La diferencia entre los dos resultados no está en los datos, que son exactamente los mismos, sino
en cuánta información extra se estuvo dispuesto a imponer. La respuesta nítida del notebook A no
venía de la muestra: venía de los seis ceros.

Mirando la inflación se ve el otro punto: baja mientras dura la restricción y se recupera en cuanto
se la suelta. Las restricciones de signo no arreglan el *price puzzle*; lo desplazan al primer
horizonte no restringido.

## 3. Ceros y signos juntos

El esquema anterior identifica un solo choque. Para ponerles nombre a los cuatro hace falta más
estructura, y aquí los dos enfoques se combinan de forma natural:

- **Ceros** para el bloque externo: ningún choque doméstico mueve los términos de intercambio al
  impacto. En una economía pequeña esto no se discute.
- **Signos** para separar los tres choques domésticos, que es donde los ceros no convencen:

| Choque | PBI | Inflación | Tasa |
|:-|:-:|:-:|:-:|
| Demanda | $+$ | $+$ | |
| Oferta | $+$ | $-$ | |
| Política monetaria | | $-$ | $+$ |

Imponer que la primera fila de $\tilde S$ sea $(s_{11}, 0, 0, 0)$ equivale a restringir $Q$ a la
forma $\operatorname{diag}(1, Q_3)$, con $Q_3$ ortonormal de $3\times 3$: el bloque externo queda
identificado recursivamente y los signos actúan solo dentro del bloque doméstico.

In [ ]:
SIGNOS = np.array([[+1, +1,  0],     # filas: PBI, inflacion, tasa
                   [+1, -1, -1],     # columnas: demanda, oferta, politica
                   [ 0,  0, +1]], float)

def asignar(D):
    '''Busca una permutacion con signo de las columnas de D que cumpla SIGNOS.'''
    cand = []
    for k in range(3):
        pedido = SIGNOS[:, k]; filas = pedido != 0
        cand.append([(j, sgn) for j in range(3) for sgn in (1, -1)
                     if np.all(np.sign(sgn * D[filas, j]) == pedido[filas])])
    usadas, salida = set(), [None] * 3
    def buscar(k):
        if k == 3:
            return True
        for j, sgn in cand[k]:
            if j in usadas:
                continue
            usadas.add(j); salida[k] = (j, sgn)
            if buscar(k + 1):
                return True
            usadas.discard(j)
        return False
    return salida if buscar(0) else None

def conjunto_cuatro(model, max_intentos=300, seed=13):
    rng = np.random.default_rng(seed)
    aceptados, intentos = [], 0
    for m in range(len(model.beta_draws)):
        B = model.reshape_beta(model.beta_draws[m], model.ncoeff_eq, n)
        L = np.linalg.cholesky(model.Sigma_draws[m])
        W = wold_weights(model, B, H)
        for _ in range(max_intentos):
            intentos += 1
            Q3, R = np.linalg.qr(rng.standard_normal((3, 3)))
            Q3 = Q3 * np.sign(np.diag(R))
            Q = np.eye(n); Q[1:, 1:] = Q3
            S0 = L @ Q
            sol = asignar(S0[1:, 1:])
            if sol is not None:
                cols = [S0[:, 0]] + [sol[k][1] * S0[:, 1 + sol[k][0]] for k in range(3)]
                aceptados.append(W @ np.column_stack(cols))
                break
    return np.array(aceptados), intentos

conjunto4, intentos4 = conjunto_cuatro(model)
print(f"aceptados {len(conjunto4)} de {len(model.beta_draws)} "
      f"({100 * len(conjunto4) / intentos4:.0f}% de los sorteos)")

impacto = np.median(conjunto4, axis=0)[0]
pd.DataFrame(impacto.round(3), index=labels,
             columns=["Externo", "Demanda", "Oferta", "Política"])

In [ ]:
# La primera fila reproduce exactamente la primera columna de Cholesky: los ceros la pinchan.
print("primera fila del impacto mediano:", impacto[0].round(3))
print("primera columna de Cholesky      :",
      np.median([np.linalg.cholesky(S)[0, 0] for S in model.Sigma_draws]).round(3))

figura_conjunto(conjunto4[:, :, :, 3], ref,
                "Choque de política monetaria: ceros para el bloque externo y signos adentro")

La estructura extra estrecha el conjunto, pero no cambia la conclusión de fondo sobre el PBI: sigue
sin estar determinado su signo. Lo que sí se gana es que los cuatro choques tienen ahora una
etiqueta económica defendible, cosa que el orden de Cholesky no daba.

## 4. La mediana no es un modelo

Un error frecuente es reportar la mediana punto a punto del conjunto como si fuera "el" resultado.
No lo es: esa mediana no corresponde en general a **ninguna** matriz de impacto del conjunto
(Fry y Pagan, 2011). Se comprueba en una línea.

In [ ]:
# Sobre una extraccion fija, tomamos muchos impactos admisibles y su mediana punto a punto.
rng = np.random.default_rng(21)
acept = []
while len(acept) < 2000:
    q = rng.standard_normal(n); q /= np.linalg.norm(q)
    s = L_d @ q
    if s[3] < 0:
        s = -s
    if s[3] > 0 and s[2] < 0:
        acept.append(s)
acept = np.array(acept)
s_med = np.median(acept, axis=0)

# Todo impacto admisible cumple ||L^{-1} s|| = 1. La mediana, no.
normas = np.linalg.norm(np.linalg.solve(L_d, acept.T), axis=0)
print(f"||L^-1 s|| de los impactos aceptados: entre {normas.min():.3f} y {normas.max():.3f}")
print(f"||L^-1 s|| de la mediana punto a punto: {np.linalg.norm(np.linalg.solve(L_d, s_med)):.3f}")
print("\nLa mediana no está en el conjunto: no es una economía, es un promedio de economías.")

Lo honesto es reportar el **rango** y decir qué fracción del conjunto sostiene cada conclusión, que
es lo que hicimos arriba con el porcentaje de modelos con PBI negativo.

Hay una advertencia adicional. Sortear $Q$ de manera uniforme no es neutral: induce un prior sobre
las respuestas que no es plano, y que puede dominar al dato cuando las restricciones son pocas
(Baumeister y Hamilton, 2015). Con dos signos al impacto, buena parte de lo que se ve en la figura
viene de ese prior implícito y no de la muestra.

## Ejercicios

1. **Alargue la restricción** a los horizontes 0 y 1, es decir exija que la inflación siga bajando
   en el trimestre siguiente. ¿Cuánto cae la tasa de aceptación? ¿Se estrecha el conjunto del PBI?
   ¿A qué horizonte se muda el *price puzzle*?

2. **Agregue un signo sobre el PBI** ($-$ al impacto) y compare con la respuesta recursiva. Ahora
   los dos esquemas coinciden en el signo: discuta si eso es un hallazgo o una tautología.

3. **Identifique el choque externo por signos** en lugar de por ceros: términos de intercambio al
   alza y PBI al alza. ¿Se parece al de Cholesky? ¿Qué dice eso sobre lo razonable del orden
   elegido en el notebook A?

4. **Reporte el conjunto de la FEVD**: para cada matriz aceptada del esquema de cuatro choques,
   calcule qué parte de la varianza del PBI a 8 trimestres explica el choque de política. ¿Cuál es
   el rango? Compárelo con el número puntual del notebook A.

5. **Fry y Pagan a mano**: entre las matrices aceptadas, busque la que minimiza la distancia a la
   mediana punto a punto (el "modelo mediano"). ¿Qué tan lejos está de la mediana?